# Reviews exploration: do money, critics and audience agree?

**Question.** Do theatrical money (`log(gross/budget)`), critic scores and audience reception point the same way — and where do they disagree?

**Sample.** Clean theatrical 100k+ movies, **year ≤ 2017** (Tag Genome has no dates / no post-2017 coverage). Optional 10k Wikidata money tier for flop coverage.

**Checks.** Associations use a **genre × decade stratified** label shuffle (global shuffle kept as secondary). Disagreement = residual of `rank(IMDb) ~ rank(log(gross/budget))`. Review text is never committed.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display, Markdown

REPO = Path.cwd()
if REPO.name == "reviews":
    REPO = REPO.parents[1]
sys.path.insert(0, str(REPO))

from analysis.reviews.build_movie_table import primary_sample
from analysis.reviews.paths import OUTPUTS, HAND_LABEL_CSV, HAND_LABEL_TEMPLATE
from analysis.reviews.plotting import apply_bw_style

apply_bw_style()
OUTPUTS.mkdir(parents=True, exist_ok=True)
MAX_YEAR = 2017

table_path = OUTPUTS / "movie_level_table.csv"
if not table_path.exists() or "disagreement_residual" not in pd.read_csv(table_path, nrows=1).columns:
    from analysis.reviews.run_exploration import main as run_main
    run_main()

movies = pd.read_csv(table_path)
primary = primary_sample(movies, max_year=MAX_YEAR)
optional = movies[movies["sample"] == "10k+_wikidata_optional"].copy()
print("100k+ clean ≤2017:", len(primary))
print("under budget:", int(primary["under_budget"].sum()),
      f"({100*primary['under_budget'].mean():.1f}%)")
print("with disagreement_residual:", int(primary["disagreement_residual"].notna().sum()))
print("disagreement_group counts:")
print(primary["disagreement_group"].value_counts(dropna=False))
primary[["imdb_id","title","year","log_gross_over_budget","imdb_rating","disagreement_residual","profit_quintile","disagreement_group"]].head()

## Movie-level table

**Measures.** Profit proxies, critic/audience scores, review counts, plus `disagreement_residual` (rank(IMDb) − fitted rank from rank(log(g/b))) for the network branch to join on `imdb_id`.

**Misses.** Residual is within-sample ranks only; not dollars of "true" profit.

In [ ]:
display(pd.read_csv(OUTPUTS / "movie_table_coverage.csv", index_col=0))
display(pd.read_csv(OUTPUTS / "disagreement_summary.csv", index_col=0))
cols = ["imdb_id","title","year","genres","log_gross_over_budget","imdb_rating",
        "disagreement_residual","profit_quintile","disagreement_group",
        "metacritic","rotten_tomatoes","n_reviews_genome"]
primary[cols].describe(include="all").T

## Idea 1 — Spearman (stratified null) and residual disagreement

**Measures.** Spearman ρ among money and scores. Primary null: shuffle within **genre × decade** strata; report null mean/std and **z-score**. Global shuffle is secondary (`p_null_global`). Disagreement = residual of `rank(IMDb) ~ rank(log(g/b))`; top 25 each direction. Display uses **extreme profit quintiles** (Q1/Q5), not break-even × median.

**Misses.** First-listed genre only for strata; 100k+ still under-represents flops.

In [ ]:
corr = pd.read_csv(OUTPUTS / "spearman_correlations.csv")
display(corr.sort_values("rho", ascending=False))
display(pd.read_csv(OUTPUTS / "idea1_summary.csv", index_col=0))

for fig in ["fig_scatter_money_scores.png", "fig_profit_quintiles_disagreement.png"]:
    p = OUTPUTS / fig
    if p.exists():
        cap = p.with_suffix(".caption.txt")
        if cap.exists():
            display(Markdown(cap.read_text()))
        display(Image(filename=str(p)))

In [ ]:
print("Top 25: rating above money (positive residual)")
display(pd.read_csv(OUTPUTS / "disagreement_top_rating_above_money.csv").head(10))
print("Top 25: rating below money (negative residual)")
display(pd.read_csv(OUTPUTS / "disagreement_top_rating_below_money.csv").head(10))

print("\nSample bias (from data):")
print(f"  100k+ ≤{MAX_YEAR}: {len(primary)} movies, "
      f"{int(primary['under_budget'].sum())} under budget "
      f"({100*primary['under_budget'].mean():.1f}%).")
if len(optional):
    ext = optional[optional["year"] <= MAX_YEAR]
    print(f"  Optional 10k Wikidata ≤{MAX_YEAR}: {len(ext)} movies, "
          f"{int(ext['under_budget'].sum())} under budget "
          f"({100*ext['under_budget'].mean():.1f}%).")

## Text features from Tag Genome reviews

**Measures.** Length; TTR on fixed 100-token samples; VADER by genre. Nulls are genre×decade stratified (global secondary).

**Misses.** VADER is not movie-specific; no review dates/stars.

In [ ]:
display(pd.read_csv(OUTPUTS / "genre_text_features.csv").head(15))
display(pd.read_csv(OUTPUTS / "text_feature_spearman.csv", index_col=0))
val = json.loads((OUTPUTS / "sentiment_validation.json").read_text())
print("Hand-label status:", val["hand_label"])
print("VADER vs Stanford:", val["stanford_vader"])
print("Hand-label CSV (gitignored text):", HAND_LABEL_CSV)
cap = OUTPUTS / "fig_text_features_genre.caption.txt"
if cap.exists():
    display(Markdown(cap.read_text()))
display(Image(filename=str(OUTPUTS / "fig_text_features_genre.png")))

## Idea 2 (exploratory) — Tag Genome language of disagreement groups

**Measures.** Term rates per 10 000 words in Tag Genome reviews for `rating_above_money` vs `rating_below_money` (extreme residual quintiles). Names removed; min term count stated in coverage CSV / caption.

**Misses.** Heuristic name filter; residual groups are rank-based, not causal types.

In [ ]:
display(pd.read_csv(OUTPUTS / "scattertext_coverage.csv", index_col=0))
display(pd.read_csv(OUTPUTS / "scattertext_top_divergent_terms.csv").head(15))
cap = OUTPUTS / "fig_scattertext_disagreement_groups.caption.txt"
if cap.exists():
    display(Markdown(cap.read_text()))
display(Image(filename=str(OUTPUTS / "fig_scattertext_disagreement_groups.png")))

## Optional extension: 10k–100k tier

**Measures.** Same profit proxy via Wikidata; Spearman with genre×decade stratified null.

**Misses.** Not hand-checked.

In [ ]:
from analysis.reviews.nulls import genre_decade_strata, spearman_with_null

optional = movies[movies["sample"] == "10k+_wikidata_optional"].copy()
if len(optional):
    ext = optional[optional["year"] <= MAX_YEAR]
    print("optional ≤2017:", len(ext),
          "under budget:", int(ext["under_budget"].sum()),
          f"({100*ext['under_budget'].mean():.1f}%)")
    res = spearman_with_null(
        ext["log_gross_over_budget"],
        ext["imdb_rating"],
        n_perm=1000,
        strata=genre_decade_strata(ext),
    )
    print("Spearman log(g/b) vs IMDb (10k optional):", res)
else:
    print("No optional rows.")

## Re-run

```bash
PYTHONPATH=. .venv/bin/python -m analysis.reviews.run_exploration
```